In [ ]:
import pandas as pd
pd.set_option("display.max_columns", 100); pd.set_option("display.width", 200)

u = pd.read_csv("../data/raw/uci/diabetic_data.csv", na_values="?")
print(u.shape)
print(u.isna().mean().round(3).sort_values(ascending=False).head(8))
print("patients:", u.patient_nbr.nunique(), "| encounters:", len(u))
print(u.discharge_disposition_id.value_counts().head(10))
print(u.readmitted.value_counts(normalize=True).round(3))
print(u.age.value_counts().sort_index())

In [ ]:
print(open("../data/raw/uci/IDS_mapping.csv").read())

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

u = pd.read_csv("../data/raw/uci/diabetic_data.csv", na_values="?", low_memory=False)

def icd9_group(code):
    if pd.isna(code): return "missing"
    c = str(code)
    if c[0] in "VE": return "other"
    try: x = float(c)
    except ValueError: return "other"
    i = int(x)
    if 390 <= x < 460 or i == 785: return "circulatory"
    if 460 <= x < 520 or i == 786: return "respiratory"
    if 520 <= x < 580 or i == 787: return "digestive"
    if i == 250: return "diabetes"
    if 800 <= x < 1000: return "injury"
    if 710 <= x < 740: return "musculoskeletal"
    if 580 <= x < 630 or i == 788: return "genitourinary"
    if 140 <= x < 240: return "neoplasms"
    return "other"

import re

def make_X(df, drop_extra=()):
    X = df.drop(columns=["encounter_id","patient_nbr","readmitted","discharge_disposition_id","weight"] + list(drop_extra))
    for c in ["diag_1","diag_2","diag_3"]:
        X[c] = X[c].map(icd9_group)
    top = X.medical_specialty.value_counts().head(15).index
    X["medical_specialty"] = X.medical_specialty.where(X.medical_specialty.isin(top), "Other").fillna("Unknown")
    for c in ["A1Cresult","max_glu_serum","payer_code","race"]:
        X[c] = X[c].fillna("Not_measured")
    for c in ["admission_type_id","admission_source_id"]:
        X[c] = X[c].astype(str)
    X = pd.get_dummies(X, drop_first=True).astype(float)
    X.columns = [re.sub(r"[\[\]<>\s,()]", "_", c) for c in X.columns]   # XGBoost-safe names
    return X

home, facility = [1,6,8], [2,3,4,5,22,23,24,30]
exp_hospice = [11,13,14,19,20,21]

gkf = GroupKFold(n_splits=5)
xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8,
                    colsample_bytree=0.8, eval_metric="logloss", random_state=42, n_jobs=-1)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))

# Target 1: facility care vs home. time_in_hospital drop (discharge ka outcome hai)
d1 = u[u.discharge_disposition_id.isin(home + facility)]
y1 = d1.discharge_disposition_id.isin(facility).astype(int)
X1 = make_X(d1, drop_extra=["time_in_hospital"])

# Target 2: readmission <30 (expired/hospice nikaal kar)
d2 = u[~u.discharge_disposition_id.isin(exp_hospice)]
y2 = (d2.readmitted == "<30").astype(int)
X2 = make_X(d2)

for name, X, y, d in [("facility vs home", X1, y1, d1), ("readmitted <30", X2, y2, d2)]:
    for mname, mdl in [("XGB", xgb), ("LogReg", lr)]:
        s = cross_val_score(mdl, X, y, cv=gkf, groups=d.patient_nbr, scoring="roc_auc")
        print(f"{name:18s} {mname:6s} n={len(d):6d} pos={y.sum():5d} ({y.mean():.1%})  AUC={s.mean():.3f} ± {s.std():.3f}")

In [ ]:
stay_cols = ["num_lab_procedures","num_procedures","num_medications","number_diagnoses"]
X1_abl = X1.drop(columns=[c for c in X1.columns if c in stay_cols])
s = cross_val_score(xgb, X1_abl, y1, cv=gkf, groups=d1.patient_nbr, scoring="roc_auc")
print(f"facility vs home, stay-counts removed  AUC={s.mean():.3f} ± {s.std():.3f}")

xgb.fit(X1, y1)
imp = pd.Series(xgb.feature_importances_, index=X1.columns).sort_values(ascending=False)
print(imp.head(12).round(3))